# 08 - FinBERT Target-Dataset Fine-Tuning

Bu notebook, mevcut `04_train_plain_sentiment_models.ipynb` deney düzenini koruyarak
`ProsusAI/finbert` modelini **aynı hedef veri seti üzerinde fine-tune eder**.

Amaç, iki farklı FinBERT koşulunu ayırmaktır:

- **Original FinBERT:** `02_evaluate_finbert_baseline.ipynb` içindeki hazır model, eğitim yok.
- **Fine-tuned FinBERT:** bu notebookta aynı train/validation/test düzeniyle yeniden eğitilen model.

## Önemli tasarım kararları

- Mevcut `plain_sentiment_v1` split dosyaları **yeniden üretilmez**, doğrudan okunur.
- Eğitim ayarları `04_train_plain_sentiment_models.ipynb` ile aynıdır:
  - seed = 42
  - epochs = 4
  - learning rate = 2e-5
  - train batch = 16
  - eval batch = 32
  - max length = 128
  - weight decay = 0.01
  - class weights = açık
- FinBERT'in hazır sınıflandırma başlığı korunur; fakat modelin özgün label sırası
  hedef projenin `negative=0, neutral=1, positive=2` sırasına güvenli biçimde yeniden eşlenir.
- Sonraki istatistiksel analizler için örnek bazlı tahminler ve sınıf olasılıkları CSV olarak kaydedilir.
- Eğitim yarıda kesilirse Hugging Face checkpoint'inden devam eder.


<!-- thesis-review-note -->
## Çalışma Notu

- Amaç: ProsusAI/finbert modelini ayni hedef splitlerle yeniden fine-tune eder.
- Girdi: Plain sentiment train/validation/test splitleri.
- Çıktı ve değerlendirme: Original FinBERT ile hedef veri setinde fine-tuned FinBERT ayrimini netlestirir.
- Sıra notu: Notebook numarasi deney akışındaki yerini gösterir; önceki numaralar tamamlanmadan kalıcı sonuç yorumları güncellenmemelidir.
- Sonuç güvenliği: Bu dosyadaki mevcut output hücreleri ve kalıcı sonuç dosyaları korunur.


In [1]:
from thesis_utils import PREVIEW_ROWS, PROJECT_ROOT, paths

from pathlib import Path
import os
import sys
import json
import gc
import inspect
import random
import subprocess

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 1) PAKET KONTROLÜ
# ============================================================
def pip_install(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except Exception:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

pip_install("datasets")
pip_install("transformers")
pip_install("torch")
pip_install("sklearn", "scikit-learn")
pip_install("accelerate")


# ============================================================
# 2) DENEY AYARLARI
# 04_train_plain_sentiment_models.ipynb ile aynı
# ============================================================
MODEL_NAME = "ProsusAI/finbert"
RUN_NAME = "finbert_target_finetuned_seed42"

RANDOM_STATE = 42
MAX_LENGTH = 128
EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 3
USE_CLASS_WEIGHTS = True

# Final model zaten varsa tekrar eğitmek yerine yükleyip değerlendirmek için False bırak.
# Baştan yeniden eğitmek istersen True yap.
FORCE_RETRAIN = False

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}
ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}
LABELS_ORDER = ["negative", "neutral", "positive"]
NUM_LABELS = 3

TEXT_COL = "input_text"
LABEL_COL = "label"
LABEL_ID_COL = "label_id"

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch:", torch.__version__)


# ============================================================
# 3) PATH'LER
# ============================================================
SPLIT_DIR = paths.PLAIN_SENTIMENT_SPLIT_V1_DIR
TRAIN_PATH = SPLIT_DIR / "train_df.parquet"
VAL_PATH = SPLIT_DIR / "val_df.parquet"
TEST_PATH = SPLIT_DIR / "test_df.parquet"

CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT
RUN_DIR = CHECKPOINT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FINAL_MODEL_DIR = RUN_DIR / "final_model"
RESULTS_DIR = paths.MODEL_RESULTS_ROOT / RUN_NAME / "training_evaluation"

for p in [CHECKPOINT_DIR, FINAL_MODEL_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("\nSplit dir     :", SPLIT_DIR)
print("Run dir       :", RUN_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Final model   :", FINAL_MODEL_DIR)
print("Results dir   :", RESULTS_DIR)


# ============================================================
# 4) SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)


Device: cpu
Torch: 2.11.0+cpu

Split dir     : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\splits\plain_sentiment_v1
Run dir       : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42
Checkpoint dir: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\checkpoints
Final model   : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\final_model
Results dir   : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results


In [2]:
# ============================================================
# 5) MEVCUT SPLITLERİ OKU
# Bu notebook split üretmez.
# Böylece 04 notebookundaki deneylerle birebir aynı test seti kullanılır.
# ============================================================
required_paths = [TRAIN_PATH, VAL_PATH, TEST_PATH]
missing = [p for p in required_paths if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Mevcut plain_sentiment_v1 split dosyaları bulunamadı.\n"
        "Önce 04_train_plain_sentiment_models.ipynb içindeki split oluşturma "
        "hücresini çalıştırmalısın.\nEksik:\n" +
        "\n".join(str(p) for p in missing)
    )

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

for name, df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}.items():
    for col in [TEXT_COL, LABEL_COL, LABEL_ID_COL]:
        if col not in df.columns:
            raise ValueError(f"{name} splitinde eksik kolon: {col}")

    df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()
    df[LABEL_ID_COL] = df[LABEL_ID_COL].astype(int)

    bad_labels = sorted(set(df[LABEL_COL]) - set(LABEL2ID))
    if bad_labels:
        raise ValueError(f"{name} splitinde geçersiz label bulundu: {bad_labels}")

    inconsistent = df[
        df.apply(lambda r: LABEL2ID[r[LABEL_COL]] != int(r[LABEL_ID_COL]), axis=1)
    ]
    if len(inconsistent) > 0:
        raise ValueError(
            f"{name} splitinde label ve label_id uyumsuzluğu var: {len(inconsistent)} satır"
        )

print("=" * 100)
print("EXISTING SPLITS LOADED")
print("=" * 100)

for name, df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}.items():
    print(f"\n{name.upper()} shape: {df.shape}")
    print(df[LABEL_COL].value_counts().sort_index())

    if "source_dataset" in df.columns:
        print("\nsource_dataset:")
        print(df["source_dataset"].value_counts(dropna=False))

# Leakage kontrolü
train_texts = set(train_df[TEXT_COL].astype(str).str.lower().str.strip())
val_texts = set(val_df[TEXT_COL].astype(str).str.lower().str.strip())
test_texts = set(test_df[TEXT_COL].astype(str).str.lower().str.strip())

print("\nLeakage check:")
print("train-val :", len(train_texts & val_texts))
print("train-test:", len(train_texts & test_texts))
print("val-test  :", len(val_texts & test_texts))

if len(train_texts & test_texts) > 0:
    raise ValueError("Train-test overlap bulundu. Deneyi başlatmadan splitleri kontrol et.")

display(train_df[[TEXT_COL, LABEL_COL, LABEL_ID_COL]].head(PREVIEW_ROWS))


EXISTING SPLITS LOADED

TRAIN shape: (12950, 24)
label
negative    1820
neutral     8137
positive    2993
Name: count, dtype: int64

source_dataset:
source_dataset
TwitterFinancialNewsSentiment    8110
FinancialPhraseBank              4840
Name: count, dtype: int64

VAL shape: (1432, 24)
label
negative    215
neutral     929
positive    288
Name: count, dtype: int64

source_dataset:
source_dataset
TwitterFinancialNewsSentiment    1432
Name: count, dtype: int64

TEST shape: (2386, 24)
label
negative     358
neutral     1549
positive     479
Name: count, dtype: int64

source_dataset:
source_dataset
TwitterFinancialNewsSentiment    2386
Name: count, dtype: int64

Leakage check:
train-val : 0
train-test: 0
val-test  : 0


,input_text,label,label_id
0,Womply 2020 State of Local Restaurants Report ...,neutral,1
1,Blackstone CEO Steve Schwarzman says the virus...,negative,0
2,Constellation Brands goes ex-dividend on Monday,neutral,1


In [3]:
# ============================================================
# 6) YARDIMCI FONKSİYONLAR
# ============================================================

def prepare_hf_dataset(split_df, tokenizer, max_length=128):
    temp = split_df[[TEXT_COL, LABEL_ID_COL]].copy()
    temp[TEXT_COL] = temp[TEXT_COL].astype(str)
    temp[LABEL_ID_COL] = temp[LABEL_ID_COL].astype(int)
    temp = temp.rename(columns={LABEL_ID_COL: "labels"})

    ds = Dataset.from_pandas(temp.reset_index(drop=True))

    def tokenize_fn(batch):
        return tokenizer(
            batch[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    ds = ds.map(tokenize_fn, batched=True)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in ds.column_names:
        keep_cols.append("token_type_ids")

    ds.set_format(type="torch", columns=keep_cols)
    return ds


def get_trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)

    if "processing_class" in sig.parameters:
        return {"processing_class": tokenizer}

    if "tokenizer" in sig.parameters:
        return {"tokenizer": tokenizer}

    return {}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


def build_training_args(output_dir):
    sig = inspect.signature(TrainingArguments.__init__)

    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        report_to="none",
        seed=RANDOM_STATE,
    )

    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "epoch"

    if "save_strategy" in sig.parameters:
        kwargs["save_strategy"] = "epoch"

    if "save_total_limit" in sig.parameters:
        kwargs["save_total_limit"] = SAVE_TOTAL_LIMIT

    if "load_best_model_at_end" in sig.parameters:
        kwargs["load_best_model_at_end"] = True

    if "metric_for_best_model" in sig.parameters:
        kwargs["metric_for_best_model"] = "f1_macro"

    if "greater_is_better" in sig.parameters:
        kwargs["greater_is_better"] = True

    if "logging_strategy" in sig.parameters:
        kwargs["logging_strategy"] = "steps"

    if "fp16" in sig.parameters:
        kwargs["fp16"] = torch.cuda.is_available()

    return TrainingArguments(**kwargs)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights_tensor is not None:
            weights = self.class_weights_tensor.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


def normalize_finbert_label(label):
    """
    ProsusAI/finbert label adlarını ortak proje formatına çevirir.
    Hazır model tipik olarak:
      0 -> positive
      1 -> negative
      2 -> neutral
    sırasını kullanır.
    """
    s = str(label).lower().strip()
    mapping = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "label_0": "positive",
        "label_1": "negative",
        "label_2": "neutral",
    }
    return mapping.get(s, s)


def align_finbert_classifier_to_project_labels(model):
    """
    FinBERT'in hazır classification head ağırlıklarını kaybetmeden,
    çıktı satırlarını proje label sırasına taşır:

        project: 0=negative, 1=neutral, 2=positive

    Bu adım kritik:
    config.label2id'i yalnızca değiştirmek, classifier weight satırlarını
    fiziksel olarak yeniden sıralamaz.
    """
    original_id2label = {
        int(k): normalize_finbert_label(v)
        for k, v in model.config.id2label.items()
    }

    print("Original FinBERT id2label:", original_id2label)

    if set(original_id2label.values()) != set(LABEL2ID.keys()):
        raise ValueError(
            "FinBERT label mapping beklenenden farklı. "
            f"Bulunan mapping: {original_id2label}"
        )

    if not hasattr(model, "classifier"):
        raise AttributeError(
            "Modelde doğrudan 'classifier' katmanı bulunamadı. "
            "ProsusAI/finbert mimarisi değişmiş olabilir."
        )

    classifier = model.classifier

    if not hasattr(classifier, "weight") or classifier.weight.shape[0] != NUM_LABELS:
        raise ValueError(
            f"Beklenmeyen classifier shape: {getattr(classifier, 'weight', None)}"
        )

    source_id_for_label = {
        label: source_id
        for source_id, label in original_id2label.items()
    }

    with torch.no_grad():
        old_weight = classifier.weight.detach().clone()
        old_bias = classifier.bias.detach().clone() if classifier.bias is not None else None

        for target_id, target_label in ID2LABEL.items():
            source_id = source_id_for_label[target_label]
            classifier.weight[target_id].copy_(old_weight[source_id])

            if old_bias is not None:
                classifier.bias[target_id].copy_(old_bias[source_id])

    model.config.id2label = ID2LABEL.copy()
    model.config.label2id = LABEL2ID.copy()

    print("Aligned project id2label:", model.config.id2label)
    return model


def calculate_class_weights(df):
    train_labels = df[LABEL_ID_COL].astype(int).values
    counts = np.bincount(train_labels, minlength=NUM_LABELS)

    weights = []
    for class_id in range(NUM_LABELS):
        if counts[class_id] == 0:
            weights.append(1.0)
        else:
            weights.append(len(train_labels) / (NUM_LABELS * counts[class_id]))

    tensor = torch.tensor(weights, dtype=torch.float)

    print("\nClass counts:", counts)
    print("Class weights:")
    for i, w in enumerate(weights):
        print(f"  {i} {ID2LABEL[i]:8s} -> {float(w):.6f}")

    return tensor


In [4]:
# ============================================================
# 7) MODEL / TOKENIZER / DATASET
# ============================================================
set_seed(RANDOM_STATE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_ds = prepare_hf_dataset(train_df, tokenizer, MAX_LENGTH)
val_ds = prepare_hf_dataset(val_df, tokenizer, MAX_LENGTH)
test_ds = prepare_hf_dataset(test_df, tokenizer, MAX_LENGTH)

final_model_exists = (
    (FINAL_MODEL_DIR / "config.json").exists()
    and any(FINAL_MODEL_DIR.glob("model*.safetensors"))
    or (FINAL_MODEL_DIR / "pytorch_model.bin").exists()
)

# Parantez önceliğini açık şekilde düzelt
final_model_exists = (
    (FINAL_MODEL_DIR / "config.json").exists()
    and (
        any(FINAL_MODEL_DIR.glob("model*.safetensors"))
        or (FINAL_MODEL_DIR / "pytorch_model.bin").exists()
    )
)

if final_model_exists and not FORCE_RETRAIN:
    print("Final model bulundu; eğitim atlanacak ve kayıtlı model değerlendirilecek.")
    print(FINAL_MODEL_DIR)

    model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
    model.config.id2label = ID2LABEL.copy()
    model.config.label2id = LABEL2ID.copy()

else:
    print("Hazır FinBERT yükleniyor:", MODEL_NAME)

    # Önce modelin özgün id2label bilgisini koruyarak yükle.
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

    # Hazır FinBERT sentiment head'ini proje label sırasına hizala.
    model = align_finbert_classifier_to_project_labels(model)

model.to(DEVICE)

class_weights_tensor = calculate_class_weights(train_df) if USE_CLASS_WEIGHTS else None

training_args = build_training_args(CHECKPOINT_DIR)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    class_weights_tensor=class_weights_tensor,
    **get_trainer_tokenizer_kwargs(tokenizer),
)

print("\nModel ready.")
print("Model config id2label:", trainer.model.config.id2label)


Map:   0%|          | 0/12950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1432 [00:00<?, ? examples/s]

Map:   0%|          | 0/2386 [00:00<?, ? examples/s]

Hazır FinBERT yükleniyor: ProsusAI/finbert


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Original FinBERT id2label: {0: 'positive', 1: 'negative', 2: 'neutral'}
Aligned project id2label: {0: 'negative', 1: 'neutral', 2: 'positive'}

Class counts: [1820 8137 2993]
Class weights:
  0 negative -> 2.371795
  1 neutral  -> 0.530499
  2 positive -> 1.442254

Model ready.
Model config id2label: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [5]:
# ============================================================
# 8) TRAIN
# ============================================================
if final_model_exists and not FORCE_RETRAIN:
    print("Training skipped: final model already exists.")

else:
    last_checkpoint = None

    if CHECKPOINT_DIR.exists() and not FORCE_RETRAIN:
        last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

    if last_checkpoint is not None:
        print("Checkpoint bulundu. Eğitim devam ediyor:")
        print(last_checkpoint)
        train_output = trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        if FORCE_RETRAIN:
            print("FORCE_RETRAIN=True -> eğitim sıfırdan başlıyor.")
        else:
            print("Checkpoint bulunamadı. Eğitim sıfırdan başlıyor.")

        train_output = trainer.train()

    print("\nTraining output:")
    print(train_output)

    print("\nFinal model kaydediliyor:")
    print(FINAL_MODEL_DIR)

    trainer.save_model(str(FINAL_MODEL_DIR))
    tokenizer.save_pretrained(str(FINAL_MODEL_DIR))


# Çalışma konfigürasyonu
run_config = {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "seed": RANDOM_STATE,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "train_path": str(TRAIN_PATH),
    "val_path": str(VAL_PATH),
    "test_path": str(TEST_PATH),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
}

with open(RUN_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print("\nRun config saved:")
print(RUN_DIR / "run_config.json")


Checkpoint bulunamadı. Eğitim sıfırdan başlıyor.


C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.435236,0.388085,0.840782,0.810587,0.846016
2,0.266172,0.452389,0.872905,0.846222,0.875253
3,0.132665,0.567715,0.871508,0.844727,0.873512
4,0.110404,0.678619,0.881983,0.853893,0.882539


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training output:
TrainOutput(global_step=3240, training_loss=0.21930032946445324, metrics={'train_runtime': 17479.9013, 'train_samples_per_second': 2.963, 'train_steps_per_second': 0.185, 'total_flos': 3407318759577600.0, 'train_loss': 0.21930032946445324, 'epoch': 4.0})

Final model kaydediliyor:
D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\final_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Run config saved:
D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\run_config.json


In [8]:
# ============================================================
# 9) DEĞERLENDİRME + ÖRNEK BAZLI PREDICTION KAYDI
# ============================================================

def evaluate_and_save(trainer, dataset, source_df, split_name):
    pred_output = trainer.predict(dataset)

    logits = pred_output.predictions
    y_true = pred_output.label_ids.astype(int)

    # Numerik kararlılık için softmax
    logits_shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits_shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

    y_pred = np.argmax(probs, axis=1).astype(int)
    confidence = probs.max(axis=1)

    # ------------------------------------------------------------
    # Güvenlik kontrolü
    # ------------------------------------------------------------
    if len(source_df) != len(y_true):
        raise ValueError(
            f"{split_name}: source_df ve prediction uzunlukları eşleşmiyor. "
            f"source_df={len(source_df)}, predictions={len(y_true)}"
        )

    # ------------------------------------------------------------
    # Genel metrikler
    # ------------------------------------------------------------
    acc = accuracy_score(y_true, y_pred)

    f1_macro = f1_score(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )

    f1_weighted = f1_score(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="weighted",
        zero_division=0,
    )

    precision_macro, recall_macro, _, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        )
    )

    per_class_p, per_class_r, per_class_f1, per_class_support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average=None,
            zero_division=0,
        )
    )

    metrics = {
        "run_name": RUN_NAME,
        "model_name": MODEL_NAME,
        "split": split_name,
        "seed": RANDOM_STATE,
        "n_eval": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),

        "precision_negative": float(per_class_p[0]),
        "recall_negative": float(per_class_r[0]),
        "f1_negative": float(per_class_f1[0]),
        "support_negative": int(per_class_support[0]),

        "precision_neutral": float(per_class_p[1]),
        "recall_neutral": float(per_class_r[1]),
        "f1_neutral": float(per_class_f1[1]),
        "support_neutral": int(per_class_support[1]),

        "precision_positive": float(per_class_p[2]),
        "recall_positive": float(per_class_r[2]),
        "f1_positive": float(per_class_f1[2]),
        "support_positive": int(per_class_support[2]),
    }

    # ------------------------------------------------------------
    # Classification report
    # ------------------------------------------------------------
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=LABELS_ORDER,
        output_dict=True,
        digits=6,
        zero_division=0,
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=LABELS_ORDER,
        digits=4,
        zero_division=0,
    )

    # ------------------------------------------------------------
    # Confusion matrix
    # ------------------------------------------------------------
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2],
    )

    row_totals = cm.sum(axis=1, keepdims=True)

    cm_norm = np.divide(
        cm,
        row_totals,
        out=np.zeros_like(cm, dtype=float),
        where=row_totals != 0,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in LABELS_ORDER],
        columns=[f"pred_{x}" for x in LABELS_ORDER],
    )

    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=[f"true_{x}" for x in LABELS_ORDER],
        columns=[f"pred_{x}" for x in LABELS_ORDER],
    )

    # ------------------------------------------------------------
    # Örnek bazlı prediction DataFrame
    # ------------------------------------------------------------
    pred_df = source_df.reset_index(drop=True).copy()

    # Dataset içinde sample_id zaten varsa onu koruyoruz.
    # Yoksa deterministic bir sample_id üretiyoruz.
    if "sample_id" not in pred_df.columns:
        pred_df.insert(
            0,
            "sample_id",
            [
                f"{split_name}_{i:06d}"
                for i in range(len(pred_df))
            ],
        )
        print(
            f"{split_name}: sample_id yoktu, "
            "otomatik olarak oluşturuldu."
        )
    else:
        # String'e çevirerek sonraki join işlemlerini daha güvenli yapıyoruz.
        pred_df["sample_id"] = (
            pred_df["sample_id"]
            .astype(str)
            .str.strip()
        )

        print(
            f"{split_name}: mevcut sample_id değerleri korunuyor."
        )

    # sample_id boş / nan kontrolü
    invalid_sample_id = (
        pred_df["sample_id"].isna()
        | pred_df["sample_id"].astype(str).str.strip().isin(
            ["", "nan", "None"]
        )
    )

    if invalid_sample_id.any():
        raise ValueError(
            f"{split_name}: "
            f"{int(invalid_sample_id.sum())} adet geçersiz sample_id bulundu."
        )

    # Duplicate ID sonraki McNemar / paired bootstrap analizlerini bozabilir.
    duplicate_count = int(
        pred_df["sample_id"].duplicated().sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{split_name}: sample_id içinde "
            f"{duplicate_count} adet duplicate değer bulundu."
        )

    # ------------------------------------------------------------
    # Gold / prediction / probability bilgileri
    # ------------------------------------------------------------
    pred_df["gold_label_id"] = y_true
    pred_df["gold_label"] = [
        ID2LABEL[int(x)]
        for x in y_true
    ]

    pred_df["prediction_id"] = y_pred
    pred_df["prediction"] = [
        ID2LABEL[int(x)]
        for x in y_pred
    ]

    pred_df["correct"] = (
        y_true == y_pred
    )

    pred_df["confidence"] = confidence

    pred_df["prob_negative"] = probs[:, 0]
    pred_df["prob_neutral"] = probs[:, 1]
    pred_df["prob_positive"] = probs[:, 2]

    # Logitleri de kaydediyoruz.
    # Sonradan calibration / ECE analizi yapmak istersek işe yarar.
    pred_df["logit_negative"] = logits[:, 0]
    pred_df["logit_neutral"] = logits[:, 1]
    pred_df["logit_positive"] = logits[:, 2]

    # Hata yönü
    pred_df["error_transition"] = np.where(
        pred_df["correct"],
        "correct",
        pred_df["gold_label"]
        + " -> "
        + pred_df["prediction"],
    )

    # ------------------------------------------------------------
    # Dosya yolları
    # ------------------------------------------------------------
    metrics_path = (
        RESULTS_DIR
        / f"{split_name}_metrics.json"
    )

    pred_path = (
        RESULTS_DIR
        / f"{split_name}_predictions.csv"
    )

    cm_path = (
        RESULTS_DIR
        / f"{split_name}_confusion_matrix.csv"
    )

    cm_norm_path = (
        RESULTS_DIR
        / f"{split_name}_confusion_matrix_normalized.csv"
    )

    report_path = (
        RESULTS_DIR
        / f"{split_name}_classification_report.csv"
    )

    # ------------------------------------------------------------
    # Kaydet
    # ------------------------------------------------------------
    with open(
        metrics_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            metrics,
            f,
            ensure_ascii=False,
            indent=2,
        )

    pred_df.to_csv(
        pred_path,
        index=False,
        encoding="utf-8-sig",
    )

    cm_df.to_csv(
        cm_path,
        encoding="utf-8-sig",
    )

    cm_norm_df.to_csv(
        cm_norm_path,
        encoding="utf-8-sig",
    )

    pd.DataFrame(
        report_dict
    ).T.to_csv(
        report_path,
        encoding="utf-8-sig",
    )

    # ------------------------------------------------------------
    # Console çıktısı
    # ------------------------------------------------------------
    print("\n" + "=" * 100)
    print(
        f"FINE-TUNED FINBERT | "
        f"{split_name.upper()}"
    )
    print("=" * 100)

    print(f"N               : {len(y_true)}")
    print(f"Accuracy        : {acc:.4f}")
    print(f"Macro Precision : {precision_macro:.4f}")
    print(f"Macro Recall    : {recall_macro:.4f}")
    print(f"Macro F1        : {f1_macro:.4f}")
    print(f"Weighted F1     : {f1_weighted:.4f}")

    print("\nClass F1:")
    print(
        f"Negative        : "
        f"{per_class_f1[0]:.4f}"
    )
    print(
        f"Neutral         : "
        f"{per_class_f1[1]:.4f}"
    )
    print(
        f"Positive        : "
        f"{per_class_f1[2]:.4f}"
    )

    print("\nClassification report:")
    print(report_text)

    print("\nConfusion matrix:")
    display(cm_df)

    print("\nNormalized confusion matrix:")
    display(
        cm_norm_df.round(4)
    )

    print("\nPrediction file:")
    print(pred_path)

    return metrics, pred_df


# ============================================================
# VALIDATION
# ============================================================

val_metrics, val_predictions_df = (
    evaluate_and_save(
        trainer,
        val_ds,
        val_df,
        "val",
    )
)


# ============================================================
# INTERNAL TEST
# ============================================================

test_metrics, test_predictions_df = (
    evaluate_and_save(
        trainer,
        test_ds,
        test_df,
        "test",
    )
)


# ============================================================
# ÖZET
# ============================================================

summary_df = pd.DataFrame(
    [
        {
            "run_name": RUN_NAME,
            "model_name": MODEL_NAME,
            "seed": RANDOM_STATE,

            "val_accuracy":
                val_metrics["accuracy"],

            "val_precision_macro":
                val_metrics["precision_macro"],

            "val_recall_macro":
                val_metrics["recall_macro"],

            "val_f1_macro":
                val_metrics["f1_macro"],

            "val_f1_weighted":
                val_metrics["f1_weighted"],

            "test_accuracy":
                test_metrics["accuracy"],

            "test_precision_macro":
                test_metrics["precision_macro"],

            "test_recall_macro":
                test_metrics["recall_macro"],

            "test_f1_macro":
                test_metrics["f1_macro"],

            "test_f1_weighted":
                test_metrics["f1_weighted"],

            "test_f1_negative":
                test_metrics["f1_negative"],

            "test_f1_neutral":
                test_metrics["f1_neutral"],

            "test_f1_positive":
                test_metrics["f1_positive"],

            "n_train":
                len(train_df),

            "n_val":
                len(val_df),

            "n_test":
                len(test_df),
        }
    ]
)


# ============================================================
# SUMMARY DOSYASI
# ============================================================

summary_path = (
    RESULTS_DIR
    / "finbert_target_finetune_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# SONUÇ
# ============================================================

print("\n" + "#" * 100)
print(
    "08 FINBERT TARGET-DATASET "
    "FINE-TUNING FINISHED"
)
print("#" * 100)

display(
    summary_df.round(6)
)

print("\nKaydedilen temel dosyalar:")

print(
    "Summary         :",
    summary_path,
)

print(
    "Test predictions:",
    RESULTS_DIR
    / "test_predictions.csv",
)

print(
    "Val predictions :",
    RESULTS_DIR
    / "val_predictions.csv",
)

print(
    "Final model     :",
    FINAL_MODEL_DIR,
)

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


val: mevcut sample_id değerleri korunuyor.

FINE-TUNED FINBERT | VAL
N               : 1432
Accuracy        : 0.8820
Macro Precision : 0.8476
Macro Recall    : 0.8605
Macro F1        : 0.8539
Weighted F1     : 0.8825

Class F1:
Negative        : 0.8356
Neutral         : 0.9158
Positive        : 0.8103

Classification report:
              precision    recall  f1-score   support

    negative     0.8206    0.8512    0.8356       215
     neutral     0.9243    0.9074    0.9158       929
    positive     0.7980    0.8229    0.8103       288

    accuracy                         0.8820      1432
   macro avg     0.8476    0.8605    0.8539      1432
weighted avg     0.8834    0.8820    0.8825      1432


Confusion matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,183,29,3
true_neutral,29,843,57
true_positive,11,40,237



Normalized confusion matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,0.8512,0.1349,0.0140
true_neutral,0.0312,0.9074,0.0614
true_positive,0.0382,0.1389,0.8229


...


Normalized confusion matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,0.8324,0.1453,0.0223
true_neutral,0.0484,0.9012,0.0504
true_positive,0.0355,0.1086,0.8559



Prediction file:
D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results\test_predictions.csv

####################################################################################################
08 FINBERT TARGET-DATASET FINE-TUNING FINISHED
####################################################################################################


,run_name,model_name,seed,val_accuracy,val_precision_macro,val_recall_macro,val_f1_macro,val_f1_weighted,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,test_f1_negative,test_f1_neutral,test_f1_positive,n_train,n_val,n_test
0,finbert_target_finetuned_seed42,ProsusAI/finbert,42,0.881983,0.84765,0.860502,0.853893,0.882539,0.881811,0.840461,0.863193,0.851176,0.882874,0.796791,0.91571,0.841026,12950,1432,2386



Kaydedilen temel dosyalar:
Summary         : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results\finbert_target_finetune_summary.csv
Test predictions: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results\test_predictions.csv
Val predictions : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\results\val_predictions.csv
Final model     : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\final_model


## Bu notebook bittikten sonra bana ne göndereceksin?

En azından son hücredeki `summary_df` çıktısını kopyalayıp gönder.

Mümkünse ayrıca şu dosyayı da yükle:

`checkpoints/financial_sentiment_multi_model/finbert_target_finetuned_seed42/results/test_predictions.csv`

Bu prediction dosyası sonraki **multi-seed** ve **istatistiksel anlamlılık** notebooklarında kullanılacak.
